# Idiap / USSD teachers: original 141 participants, seed 103

Run cells in order. This notebook starts with the **split and provenance audit**. It uses the original labeled TRAIN-107 and DEV-35, excludes corrupt participant 440, then creates participant-disjoint TRAIN-113 / VAL-14 / internal holdout-14. The official TEST roster, files, and labels are never opened. The internal holdout remains closed until the method is frozen.

**Checkpoint rule:** The published Idiap and USSD checkpoints already saw participants from the original training split. They cannot be used to evaluate this redistributed split without exposure leakage. Train fresh instances of their architectures on TRAIN-113. The older released checkpoints and original experiment folders remain untouched.

Stage 1 below saves only the split. Later teacher stages will be added here after the split audit is confirmed; rerun the checkout cell to load new revisions.


## 1. Mount Drive and load the pinned GitHub branch


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
import base64, os, subprocess, sys
DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
assert (DAIC / 'metadata').is_dir(), f'Missing metadata under {DAIC}'
REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
BRANCH = 'feature/official141-original-teachers-80-10-10'
ROOT = Path('/content/reliability-aware-depression-kd')
token = userdata.get('GITHUB_TOKEN'); assert token, 'Add GITHUB_TOKEN to Colab Secrets'
header = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy(); env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0='http.extraHeader', GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {header}')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', f'https://github.com/{REPO}.git', str(ROOT)], check=True, env=env)
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH], check=True, env=env)
revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'FETCH_HEAD'], text=True).strip()
CODE = Path('/content/reliability-aware-depression-kd-official141-' + revision[:12])
if not CODE.exists(): subprocess.run(['git', '-C', str(ROOT), 'worktree', 'add', '--detach', str(CODE), revision], check=True)
assert (CODE / '.git').exists() and subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', 'HEAD'], text=True).strip() == revision
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements-colab.txt')], check=True)
del token, header, env
print('Revision:', revision)


## 2. Freeze the 113 / 14 / 14 split


In [ ]:
import pandas as pd
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from scripts.experiments.official141.split import main as split_stage
EXP = DAIC / 'experiments' / 'official141_80_10_10_seed103'
split_stage(['--daic-root', str(DAIC), '--output', str(EXP / 'split'), '--seed', '103'])
manifest = pd.read_csv(EXP / 'split' / 'manifest.csv')
assert len(manifest) == 141 and manifest.participant_id.nunique() == 141
assert 440 not in set(manifest.participant_id)
print(pd.crosstab(manifest.split, manifest.label))
print('Source audit:', manifest.groupby(['split', 'source_split']).size().to_dict())
print('Saved:', EXP / 'split' / 'manifest.csv')


## Stop after the split audit

Share the printed class-count and source tables. The next stage will validate transcripts and audio coverage for these exact IDs, then train and evaluate the text teacher, followed by the audio teacher. No official TEST access and no OOF.
